# Credit Risk: Vintage Curves & Roll-Rate Transition Matrix

Read-only, exploratory. Plots the two core outputs of the credit-risk
practice module — see `dbt/models/credit_risk/README.md` for the full
writeup of what these are and how they map to real CECL loss forecasting.

Assumes `dbt build` (or `make build`) has already run, so
`main_marts.credit_risk__*` tables exist. Per `notebooks/AGENTS.md`, this
notebook only reads — it never builds or refreshes anything.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/Users/trustanprice/Desktop/Personal/ledgerone")
DB_PATH = PROJECT_ROOT / "data/processed/ledgerone.duckdb"

con = duckdb.connect(str(DB_PATH), read_only=True)

## Vintage curves: cumulative charge-off rate by cohort x months-on-book

Each line is one origination quarter. Later cohorts are shorter lines —
that's right-censoring (a 2024-Q4 cohort hasn't had time to season yet),
not missing data. Compare cohorts at the same months-on-book, not the same
calendar date.

In [ ]:
vintage = con.execute("""
    SELECT origination_quarter, months_on_book, cumulative_charge_off_rate
    FROM main_marts.credit_risk__vintage_curves
    ORDER BY origination_quarter, months_on_book
""").df()

fig, ax = plt.subplots(figsize=(9, 5.5))
for cohort, grp in vintage.groupby("origination_quarter"):
    ax.plot(grp["months_on_book"], grp["cumulative_charge_off_rate"] * 100, label=str(cohort)[:7])
ax.set_title("Cumulative charge-off rate by vintage cohort")
ax.set_xlabel("Months on book")
ax.set_ylabel("Cumulative charge-off rate (%)")
ax.legend(fontsize=7, ncol=2, title="Origination quarter")
fig.tight_layout()
plt.show()

## Roll-rate transition matrix (blended across the full observation window)

Row-normalized: each row sums to 1. Reading a row: "of accounts in
`from_bucket` this month, what share ended up in each `to_bucket` next
month?"

In [ ]:
roll_rate = con.execute("""
    SELECT from_bucket, to_bucket, SUM(account_count) AS account_count
    FROM main_marts.credit_risk__roll_rate_transition_matrix
    GROUP BY 1, 2
""").df()
roll_rate["transition_rate"] = roll_rate.groupby("from_bucket")["account_count"].transform(lambda x: x / x.sum())

bucket_order = ["Current", "30-59 DPD", "60-89 DPD", "90-119 DPD", "120+/Charged-Off"]
matrix = (
    roll_rate.pivot(index="from_bucket", columns="to_bucket", values="transition_rate")
    .reindex(index=bucket_order[:-1], columns=bucket_order)
)
matrix.style.background_gradient(cmap="Blues", axis=1).format("{:.1%}", na_rep="—")

## CECL-style 12-month reserve estimate

Rolling the matrix above forward — see `credit_risk__cecl_reserve_estimate`
and its docs for how.

In [ ]:
reserve = con.execute("SELECT * FROM main_marts.credit_risk__cecl_reserve_estimate").df()
display(reserve)

total_balance = reserve["outstanding_balance"].sum()
total_expected_loss = reserve["expected_loss_amount"].sum()
print(f"Blended reserve rate: {total_expected_loss / total_balance:.2%}")

con.close()